## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-14-09-30-58 +0000,chitri,Today in History: Nellie Bly travels around th...,https://www.chicagotribune.com/2026/11/14/toda...
1,2026-11-14-09-28-56 +0000,nypost,Microsoft restores service to Outlook and Team...,https://nypost.com/2026/11/14/business/microso...
2,2026-11-14-09-15-41 +0000,nypost,Bomb threat at Illinois ethanol plant forces m...,https://nypost.com/2026/11/14/us-news/illinois...
3,2026-11-14-09-10-40 +0000,nyt,"Bomb Found at Ethanol Plant in Peoria, Ill., M...",https://www.nytimes.com/2026/11/14/us/bomb-eth...
4,2026-11-14-09-05-48 +0000,bbc,Young people should have legal guarantee of jo...,https://www.bbc.co.uk/news/articles/c69vwp09d7...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2437/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
426,nvidia,32
133,trump,29
388,huang,21
359,new,21
62,ground,20


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
59,2026-11-14-02-07-00 +0000,wsj,"More than almost any CEO, Jensen Huang helped ...",https://www.wsj.com/politics/policy/nvidia-jen...,138
294,2026-11-13-16-01-00 +0000,wsj,Treasury Secretary Scott Bessent said Howard L...,https://www.wsj.com/politics/policy/bessent-sa...,125
392,2026-11-13-10-00-00 +0000,wapo,How disgraced Nvidia CEO Jensen Huang became a...,https://www.washingtonpost.com/technology/inte...,117
114,2026-11-13-23-12-22 +0000,nypost,Jensen Huang commemorative plaque gone from fa...,https://nypost.com/2026/11/13/business/jensen-...,101
65,2026-11-14-02-00-00 +0000,wsj,The White House ordered officials this summer ...,https://www.wsj.com/politics/policy/china-vape...,95


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
59,138,2026-11-14-02-07-00 +0000,wsj,"More than almost any CEO, Jensen Huang helped ...",https://www.wsj.com/politics/policy/nvidia-jen...
313,89,2026-11-13-14-55-01 +0000,nypost,Treasury Secretary Scott Bessent points finger...,https://nypost.com/2026/11/13/us-news/scott-be...
102,78,2026-11-13-23-52-22 +0000,latimes,"LAX, other airports filled with stranded passe...",https://www.latimes.com/california/story/2026-...
65,66,2026-11-14-02-00-00 +0000,wsj,The White House ordered officials this summer ...,https://www.wsj.com/politics/policy/china-vape...
201,62,2026-11-13-20-07-00 +0000,wsj,Rep. Lauren Boebert lost her deep-red Colorado...,https://www.wsj.com/politics/elections/lauren-...
183,58,2026-11-13-21-01-00 +0000,wsj,Of all the ways Chairman Kevin Warsh wants to ...,https://www.wsj.com/economy/central-banking/fe...
173,57,2026-11-13-21-15-24 +0000,bbc,Watch: Cruise ship seen sinking after collisio...,https://www.bbc.co.uk/news/videos/ck4gn4y77m40...
374,55,2026-11-13-10-55-43 +0000,cbc,Microsoft reports progress in reviving some se...,https://www.cbc.ca/news/world/microsoft-outage...
61,53,2026-11-14-02-03-00 +0000,wsj,California voters had a chance to impose the n...,https://www.wsj.com/politics/california-voters...
177,52,2026-11-13-21-06-14 +0000,nypost,"Trump will have historic meeting with Xi, Puti...",https://nypost.com/2026/11/13/us-news/trump-wi...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
